# Lab 2.2 &mdash; ReAct, and the Contract That Actually Breaks

**Level:** Intermediate &rarr; Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 1 &middot; Module 2 &mdash; Agentic Planning &amp; Reasoning**

### What you'll do
- Parse the classic Thought / Action / Action Input text format
- Watch the format hold while the <i>argument</i> drifts under an aged prompt
- Move the contract into an <code>args_schema</code> the model is shown upfront

> **How this lab works.** You write real LangChain and LangGraph code. Fill every `BLANK`,
> then run the **Self-check** cell under each section &mdash; those check the *objects you built*
> (a bound tool, a compiled graph, an emitted tool call), so they are deterministic and do not
> depend on the model. Cells marked **Run it for real** put your code in front of the sandbox
> model; that is the part worth watching. The score line is feedback, not a grade.

> **The thread.** All five Module 2 labs work one case: an internal employee help desk.
> The rules are ordinary on purpose &mdash; the only new thing here is how the agent reasons.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-2-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError as exc:
        print(f"(a blank above is still unfilled: {exc} -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nSelf-check: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

# The served model reasons before it answers, and the reasoning is billed as completion
# tokens: 24.1s / 980 tokens with it on, 0.7s / 29 with it off, for the same answer. Off is
# the default here because you will make a lot of calls today. Pass think=True to see the
# difference for yourself -- and note that prompts written as an explicit ordered procedure
# survive thinking being off, while vague ones do not.
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm_cache = {}
def get_llm(temperature: float = 0.0, think: bool = False):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    from langchain_openai import ChatOpenAI
    key = (temperature, think)
    if key not in _llm_cache:
        kwargs = {} if think else {"extra_body": NO_THINK}
        _llm_cache[key] = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                                     api_key=LLM_API_KEY, temperature=temperature, **kwargs)
    return _llm_cache[key]

def ask(prompt: str, system: str | None = None, think: bool = False) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm(think=think).invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

def show_messages(messages, width: int = 88) -> None:
    """Print a message list the way a trace reads: type, content, and any tool calls."""
    for m in messages:
        kind = getattr(m, "type", "?")
        body = str(getattr(m, "content", "")).replace("\n", " ")[:width]
        calls = getattr(m, "tool_calls", None)
        line = f"  [{kind:9}] {body}"
        if calls:
            line += "  -> calls: " + ", ".join(f"{c['name']}({c['args']})" for c in calls)
        print(line)

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- the object-level self-checks still work)")

In [ ]:
# ------------------------------------------------ the case file (synthetic, self-contained)
# An internal employee help desk. Ordinary rules on purpose: the only new thing in these five
# labs is LangChain. Nothing here is real data and nothing leaves this notebook.

REQUESTS = {
    "EHD-7001": {"who": "Priya Nair",   "category": "access",   "urgency": "high",
                 "wants": "reset",
                 "text": "Locked out of the payroll portal after the password reset."},
    "EHD-7002": {"who": "Rahul Menon",  "category": "hardware", "urgency": "high",
                 "wants": "replacement",
                 "text": "Laptop battery has swollen and the case is bulging."},
    "EHD-7003": {"who": "Anita Sharma", "category": "software", "urgency": "low",
                 "wants": "licence",
                 "text": "Need a licence for the diagramming tool, about 180 USD a year."},
    "EHD-7004": {"who": "Vikram Rao",   "category": "access",   "urgency": "medium",
                 "wants": "admin-rights",
                 "text": "Please give me admin rights on the finance reporting system."},
    "EHD-7005": {"who": "Priya Nair",   "category": "hardware", "urgency": "low",
                 "wants": "replacement",
                 "text": "Second monitor flickers every few minutes."},
}

# The handbook, one entry per category. Every judgement in this module comes from these.
HANDBOOK = {
    "access":   "Verify identity, then reset. The help desk NEVER grants elevated or admin "
                "rights -- route those to Identity and Access Management.",
    "hardware": "Replace under warranty. A swollen battery is a safety issue: stop use "
                "immediately and replace the same day, whatever urgency the employee set.",
    "software": "Licences over 100 USD per year need the cost-centre owner's approval first.",
}

SLA_HOURS = {"high": 4, "medium": 24, "low": 72}
ROUTE_OUT = {"admin-rights"}     # what the help desk must hand to another team, never do itself

print(f"{len(REQUESTS)} help desk requests, {len(HANDBOOK)} handbook entries loaded")

## Concept

**ReAct** interleaves reasoning and acting: *Thought* (what do I know?), *Action* (what shall I
do?), *Observation* (what came back?), repeat.

The original formulation asks the model to emit that shape **as text**, so you have to parse it.
Everyone expects the parse to be the fragile part. On this sandbox it is not: measured over four
requests, the format parsed 4/4 strictly and 4/4 loosely, twice.

What broke was the **argument**. The same agent went from 4/4 usable to 0/4 &mdash; same format,
same parser, same tool &mdash; because the model started passing a description where a reference
belongs. This lab reproduces that, and then fixes it in the one place that works.

## Section 1 &mdash; The format, and what the parser does when it is wrong

The happy path:

```
Thought: I need the stored request first.
Action: lookup_request
Action Input: EHD-7002
```

The regex is given &mdash; it is a Python idiom, not the lesson. The decision is the branch below
it: the text parsed cleanly and named a tool that **does not exist**. A parser can do three
things there. It can `raise`, which ends the run mid-turn and throws away a thought the model has
already paid for. It can return `None`, which tells the agent loop nothing at all. Or it can hand
back something the loop can act on next turn.

In [ ]:
import re
from langchain_core.tools import tool
from pydantic import BaseModel, Field

STEP_RE = re.compile(
    r"Thought:\s*(?P<thought>.*?)\s*"
    r"Action:\s*(?P<action>[\w_]+)\s*"
    r"Action Input:\s*(?P<input>.*?)\s*$",
    re.DOTALL | re.IGNORECASE)

TOOLS = {"lookup_request", "handbook_rule"}      # every tool this agent actually has


def parse_step(text: str):
    """-> {"thought", "action", "input"} for an action step, or None for a final answer."""
    m = STEP_RE.search(text or "")
    if not m:
        return None
    step = {"thought": m.group("thought").strip(),
            "action": m.group("action").strip(),
            "input": m.group("input").strip().strip('"').strip("'")}

    if step["action"] not in TOOLS:
        # The text parsed. The model named a tool that is not on the list.
        give_up        = None
        tell_the_agent = {**step, "error": f'no tool named "{step["action"]}". '
                                           f'Available: {", ".join(sorted(TOOLS))}'}
        return tell_the_agent   # an observation the model can correct from next turn

    return step

In [ ]:
# --- Self-check: Section 1   (pure string work -- no model)
GOOD   = "Thought: I need the record.\nAction: lookup_request\nAction Input: EHD-7002"
QUOTED = 'Thought: t\nAction: lookup_request\nAction Input: "EHD-7002"'
NOSUCH = "Thought: I will check the ticket.\nAction: fetch_ticket\nAction Input: EHD-7002"

check("a well-formed step parses and names the tool",
      lambda: parse_step(GOOD)["action"] == "lookup_request")
check("the argument comes out without its quotes",
      lambda: parse_step(QUOTED)["input"] == "EHD-7002")
check("a final answer is not an action step",
      lambda: parse_step("Replace the laptop the same day.") is None)
check("an unknown tool still MATCHES the format -- the format was never the problem",
      lambda: STEP_RE.search(NOSUCH) is not None)
check("...and the parser hands the loop something it can act on",
      lambda: "fetch_ticket" in parse_step(NOSUCH)["error"]
              and "lookup_request" in parse_step(NOSUCH)["error"],
      "None tells the loop nothing; raising ends the run and bins the thought")
score()

## Section 2 &mdash; Where the contract belongs

Now the failure a parser cannot catch. Under a **fresh** prompt the model passes `EHD-7002`.
Under an **aged** prompt &mdash; a few turns of history in which requests were discussed by
description &mdash; it starts passing `the swollen battery one`, or `Rahul Menon`, or
`{"request_id": "EHD-7002"}`. Every one of those parses. None of them resolves.

You can chase that in the parser: reject anything that is not `EHD-` plus four digits, and retry
the turn. That works and it is a correction applied *after* the model has spoken, once per drift,
forever.

Or you give the tool a typed signature. `args_schema` is not validation for your benefit &mdash;
LangChain sends that schema, field descriptions and all, to the model **with the request**. The
model never sees your function; it sees a name, a description and a shape.

In [ ]:
def lookup_args_schema():
    """The model READS this description before it answers. That is what it is for."""
    names_the_field = "The request to look up."
    names_the_shape = ("The help desk reference exactly as it appears on the request, "
                       "e.g. EHD-7002. Not the employee's name and not the request text.")

    class LookupArgs(BaseModel):
        request_id: str = Field(description=names_the_shape)

    return LookupArgs


def build_lookup_tool():
    @tool("lookup_request", args_schema=lookup_args_schema())
    def lookup_request(request_id: str) -> str:
        """Look up one stored help desk request by its reference."""
        r = REQUESTS.get(request_id)
        return "no such request" if r is None else f'{r["who"]} | {r["category"]} | {r["text"]}'
    return lookup_request


def where_the_contract_belongs() -> str:
    stricter_parser = "reject any Action Input that is not EHD-nnnn, and retry the turn"
    typed_signature = "give the tool an args_schema, so the model is told the shape upfront"
    return typed_signature   # shown before the answer beats corrected after it

In [ ]:
# --- Self-check: Section 2   (a real @tool and its real args_schema -- no model)
def field_description():
    return lookup_args_schema().model_fields["request_id"].description

check("the tool carries a typed args_schema",
      lambda: build_lookup_tool().args_schema is not None)
check("the description the model reads names the SHAPE of the reference",
      lambda: "EHD-7002" in field_description(),
      '"The request to look up." leaves the model to invent a format, which it will')
check("it also rules out the two things the model reaches for instead",
      lambda: "name" in field_description().lower() and "text" in field_description().lower())
check("the tool resolves a reference and refuses a description",
      lambda: "Rahul" in build_lookup_tool().invoke({"request_id": "EHD-7002"})
              and build_lookup_tool().invoke({"request_id": "the swollen battery one"})
                  == "no such request")
check("the contract belongs where the model can read it before it answers",
      lambda: where_the_contract_belongs().startswith("give the tool an args_schema"))
score()

## Run it for real

Three runs over the same four requests: text ReAct on a fresh prompt, text ReAct on an aged one,
and the bound tool on the aged one. Watch `parsed` stay flat while `usable` collapses.

In [ ]:
RIDS  = ["EHD-7001", "EHD-7002", "EHD-7003", "EHD-7005"]
FRESH = ("Answer with exactly one step, in this format and nothing else:\n"
         "Thought: <why>\nAction: <tool name>\nAction Input: <the argument>\n"
         "Tools: lookup_request(request_id), handbook_rule(category).")
AGED  = FRESH + ("\n\nEarlier in this conversation you handled the payroll lockout, the "
                 "flickering monitor and the licence for the diagramming tool. Keep referring "
                 "to requests the way the employee described them.")

def usable(step) -> bool:
    """Usable means the argument is a reference the tool can actually resolve."""
    return bool(step) and "error" not in step and step["input"] in REQUESTS

def run_text_arm(label, system):
    print(f"=== text ReAct, {label} ===")
    parsed = ok = 0
    for rid in RIDS:
        step = parse_step(ask(f'Employee request: "{REQUESTS[rid]["text"]}"\n'
                              "Look up the stored request.", system=system))
        parsed += step is not None
        ok += usable(step)
        print(f'  {rid}: input={(step or {}).get("input")!r}')
    print(f"  -> parsed {parsed}/{len(RIDS)}, usable {ok}/{len(RIDS)}\n")

def compare():
    run_text_arm("fresh prompt", FRESH)
    run_text_arm("aged prompt", AGED)

    print("=== bound tool with an args_schema, aged prompt ===")
    bound = get_llm().bind_tools([build_lookup_tool()])
    ok = 0
    for rid in RIDS:
        calls = bound.invoke([("system", AGED),
                              ("human", f'Employee request: "{REQUESTS[rid]["text"]}" '
                                        "-- look up the stored request.")]).tool_calls
        arg = calls[0]["args"].get("request_id") if calls else None
        ok += arg in REQUESTS
        print(f'  {rid}: request_id={arg!r}')
    print(f"  -> usable {ok}/{len(RIDS)}")

if llm_ready():
    guard(compare)

### Read it

The parse counts barely moved. The usable counts did &mdash; the aged prompt did not corrupt the
format, it corrupted the argument, and a regex that checks the shape of the *text* has nothing to
say about that. Every step it rejected was a step the model was entitled to believe was fine.

The bound arm got the same aged prompt and the same drifting history, and passed the reference
anyway. Nothing corrected it. The schema went out **with** the question, so there was no wrong
answer to correct.

That is the general shape, and it is worth carrying into Module 4: a contract the model is
**shown** beats a parser that corrects it afterwards. The parser was never the problem.

In [ ]:
score()

## Your turn

1. Set the `request_id` description back to `names_the_field` and re-run the bound arm on the
   aged prompt. The schema is still there and still typed. How much of the fix was the type?
2. Add `handbook_rule` as a second bound tool with a `category` field whose description does
   *not* list the three valid categories. Count how often the model invents a fourth.